# Qwen 2B LoRA Fine-Tuning for Text-to-SVG Generation (Colab A100)

Fine-tunes `Qwen3.5-2B-Instruct` with LoRA on your CSV dataset of (prompt, svg) pairs,
then generates 1000 SVGs from test prompts (max 8000 chars each) for Kaggle submission.

**Optimized for:** Google Colab with A100 GPU + High RAM runtime

**Setup:** Runtime → Change runtime type → A100 GPU, High RAM

## 1. Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

# Unsloth's recommended Colab install
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

!uv pip install -qqq \
    "torch>=2.5" "triton>=3.1" {_numpy} {_pil} torchvision bitsandbytes xformers \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"

!uv pip install --upgrade --no-deps tokenizers trl unsloth unsloth_zoo
!uv pip install datasets pandas lxml

## 2. Upload Your Data

Upload your training CSV and test prompts CSV to Colab. You can either:
- Use the file browser (left sidebar) to upload directly
- Mount Google Drive

Update the paths in CONFIG below accordingly.

In [ ]:
# Option A: Mount Google Drive (recommended for large files)
from google.colab import drive
drive.mount('/content/drive')

# Then set paths like:
# "train_csv": "/content/drive/MyDrive/your_folder/train_data.csv",
# "test_prompts_csv": "/content/drive/MyDrive/your_folder/test_prompts.csv",

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Configuration

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
BF16 support: True


In [ ]:
# ── All configurable paths and hyperparameters ──
CONFIG = {
    # ═══ PATHS — UPDATE THESE ═══
    "train_csv": "/content/drive/MyDrive/train_clean.csv",       # Your CSV with 'prompt' and 'svg' columns
    "test_prompts_csv": "/content/drive/MyDrive/test.csv", # Test CSV with 'id' and 'prompt' columns
    "output_dir": "/content/qwen2b_svg_lora",
    "submission_path": "/content/submission.csv",

    # Model
    "model_name": "unsloth/Qwen3.5-2B",
    "max_seq_length": 1024,
    "load_in_4bit": True,

    # LoRA
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0,

    # Training — tuned for A100
    "max_steps": 500,                    # ~1500 steps is enough to learn SVG structure
    "per_device_train_batch_size": 32,      # A100 80GB handles this easily
    "gradient_accumulation_steps": 2,      # Effective batch = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "save_steps": 100,
    "eval_size": 0.01,

    # Inference
    "max_new_tokens": 4096,
    "max_svg_chars": 8000,
    "temperature": 0.6,
    "top_p": 0.9,
    "repetition_penalty": 1.05,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Config:
  train_csv: /content/drive/MyDrive/train_clean.csv
  test_prompts_csv: /content/drive/MyDrive/test.csv
  output_dir: /content/qwen2b_svg_lora
  submission_path: /content/submission.csv
  model_name: unsloth/Qwen3.5-2B
  max_seq_length: 1024
  load_in_4bit: True
  lora_r: 32
  lora_alpha: 64
  lora_dropout: 0
  max_steps: 500
  per_device_train_batch_size: 32
  gradient_accumulation_steps: 2
  learning_rate: 0.0002
  warmup_ratio: 0.05
  weight_decay: 0.01
  logging_steps: 10
  save_steps: 100
  eval_size: 0.01
  max_new_tokens: 4096
  max_svg_chars: 8000
  temperature: 0.6
  top_p: 0.9
  repetition_penalty: 1.05


## 4. Load and Prepare Dataset

In [ ]:
# ── Load your CSV ──
df = pd.read_csv(CONFIG["train_csv"])
print(f"Raw CSV rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst row preview:")
print(f"  prompt: {str(df['prompt'].iloc[0])[:120]}...")
print(f"  svg length: {len(str(df['svg'].iloc[0]))} chars")

Raw CSV rows: 33000
Columns: ['prompt', 'svg']

First row preview:
  prompt: The image displays a black icon with a photo-like rectangle containing a wavy line on the left side and a plus sign on t...
  svg length: 2333 chars


In [ ]:
# ── Clean and filter ──
df = df.dropna(subset=["prompt", "svg"])
df["prompt"] = df["prompt"].astype(str).str.strip()
df["svg"] = df["svg"].astype(str).str.strip()

# Keep only valid-looking SVGs within length limit
df = df[df["svg"].str.lower().str.startswith("<svg")]
df = df[df["prompt"].str.len() > 0]
df["svg_len"] = df["svg"].str.len()
df = df[df["svg_len"] <= CONFIG["max_svg_chars"]]

print(f"After cleaning: {len(df)} rows")
print(f"\nSVG length stats:")
print(df["svg_len"].describe())
print(f"\nPercentiles:")
print(df["svg_len"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

After cleaning: 33000 rows

SVG length stats:
count    33000.000000
mean      2218.354667
std       1651.384960
min         91.000000
25%        929.000000
50%       1799.000000
75%       3097.250000
max       7999.000000
Name: svg_len, dtype: float64

Percentiles:
0.50    1799.00
0.75    3097.25
0.90    4627.00
0.95    5664.05
0.99    7253.02
Name: svg_len, dtype: float64


In [ ]:
# ── Format into chat template for SFT ──
SYSTEM_PROMPT = (
    "You are an SVG generation assistant. Given a text description, generate valid SVG code. "
    "Output ONLY the SVG markup starting with <svg> and ending with </svg>. "
    "Use a 256x256 canvas with viewBox='0 0 256 256'. Keep the SVG compact and under 8000 characters."
)


def format_sft_text(row):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['svg']}<|im_end|>"
    )


df["text"] = df.apply(format_sft_text, axis=1)

# Token length estimates
df["est_tokens"] = df["text"].str.len() / 3.5
over_limit = (df["est_tokens"] > CONFIG["max_seq_length"]).sum()
print(f"Samples likely exceeding {CONFIG['max_seq_length']} tokens: {over_limit}")
print(f"These will be truncated during training.")

Samples likely exceeding 1024 tokens: 8156
These will be truncated during training.


In [ ]:
# ── Create HuggingFace Dataset and split ──
dataset = Dataset.from_pandas(df[["text"]].reset_index(drop=True))
dataset = dataset.shuffle(seed=SEED)

splits = dataset.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples:  {len(eval_dataset)}")

Train samples: 32670
Eval samples:  330


## 5. Load Model + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=CONFIG["load_in_4bit"],
)

print(f"Model loaded: {CONFIG['model_name']}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Model loaded: unsloth/Qwen3.5-2B


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
Trainable: 21,823,488 / 1,397,711,680 (1.56%)


## 6. Training

In [ ]:
from transformers import TrainingArguments

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=SFTConfig(
        output_dir=CONFIG["output_dir"],
        max_steps=CONFIG["max_steps"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        warmup_steps=50,
        weight_decay=CONFIG["weight_decay"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=CONFIG["logging_steps"],
        eval_strategy="steps",
        eval_steps=25,
        save_steps=CONFIG["save_steps"],
        save_total_limit=2,
        report_to="none",
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        seed=SEED,
        dataloader_num_workers=4,
    ),
)

print(f"Trainer ready. max_steps={CONFIG['max_steps']}")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/32670 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/330 [00:00<?, ? examples/s]

Trainer ready. max_steps=500


In [ ]:
# ── Memory before training ──
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1e9, 2)
max_memory = round(gpu_stats.total_memory / 1e9, 2)
print(f"GPU: {gpu_stats.name} | VRAM: {max_memory} GB | Reserved: {start_gpu_memory} GB")

GPU: NVIDIA A100-SXM4-80GB | VRAM: 85.09 GB | Reserved: 4.26 GB


In [ ]:
train_result = trainer.train()

# ── Stats ──
used_memory = round(torch.cuda.max_memory_reserved() / 1e9, 2)
runtime_min = train_result.metrics['train_runtime'] / 60
print(f"\nDone! {runtime_min:.1f} min | Peak VRAM: {used_memory} GB")
print(f"Train loss: {train_result.metrics.get('train_loss', 'N/A')}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,670 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 21,823,488 of 2,235,065,152 (0.98% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,0.542393,0.442799
50,0.395892,0.388223
75,0.387987,0.365953
100,0.365385,0.352044
125,0.368032,0.345354
150,0.357710,0.338828
175,0.351461,0.335430
200,0.344195,0.331545
225,0.334898,0.328235
250,0.338311,0.325187



Done! 127.8 min | Peak VRAM: 46.9 GB
Train loss: 0.36303965759277346


In [ ]:
# ── Save LoRA adapter ──
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"LoRA adapter saved to: {CONFIG['output_dir']}")

LoRA adapter saved to: /content/qwen2b_svg_lora


## 7. Inference Utilities

In [ ]:
# ── SVG extraction and validation ──
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask", "linearGradient", "radialGradient",
    "stop", "text", "tspan", "title", "desc", "style", "pattern", "marker", "filter",
    "clippath", "lineargradient", "radialgradient",
}


def extract_svg(text):
    m = SVG_REGEX.search(text)
    if not m:
        return ""
    svg = m.group(0).strip()
    if len(svg) > CONFIG["max_svg_chars"]:
        return ""
    return svg


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    local_name = root.tag.split("}")[-1] if "}" in root.tag else root.tag
    if local_name.lower() != "svg":
        return False
    for elem in root.iter():
        tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
        if tag.lower() not in ALLOWED_TAGS:
            return False
    return True


def ensure_svg_header(svg_text):
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if 'viewBox' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    return svg_text


def fallback_svg(prompt):
    h = hash(prompt) % 360
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="80" fill="hsl({h}, 60%, 50%)"/>'
        '</svg>'
    )


print("Inference utilities loaded.")

Inference utilities loaded.


In [ ]:
FastLanguageModel.for_inference(model)

text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer

SYSTEM_PROMPT = (
    "You are an SVG generation assistant. Given a text description, generate valid SVG code. "
    "Output ONLY the SVG markup. Always use: "
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">. '
    "Use only allowed elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, "
    "title, desc, style, pattern, marker, filter. "
    "Keep the SVG under 8000 characters."
)

max_new_tokens=1024

def fix_svg_canvas(svg):
    """Force 256x256 canvas on all SVGs."""
    svg = re.sub(r'width="[^"]*"', 'width="256"', svg, count=1)
    svg = re.sub(r'height="[^"]*"', 'height="256"', svg, count=1)
    svg = re.sub(r"width='[^']*'", "width='256'", svg, count=1)
    svg = re.sub(r"height='[^']*'", "height='256'", svg, count=1)
    svg = re.sub(r'viewBox="[^"]*"', 'viewBox="0 0 256 256"', svg, count=1)
    svg = re.sub(r"viewBox='[^']*'", "viewBox='0 0 256 256'", svg, count=1)
    if 'viewBox' not in svg:
        svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    if 'width=' not in svg:
        svg = svg.replace('<svg', '<svg width="256" height="256"', 1)
    if 'xmlns=' not in svg:
        svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    return svg


def generate_svg(prompt, max_new_tokens=2048):
    chat_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    input_ids = text_tokenizer.encode(chat_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    eos_ids = text_tokenizer.encode("<|im_end|>", add_special_tokens=False)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=CONFIG["temperature"],
            top_p=CONFIG["top_p"],
            repetition_penalty=CONFIG["repetition_penalty"],
            eos_token_id=eos_ids,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    decoded = text_tokenizer.decode(new_tokens, skip_special_tokens=True)

    svg = extract_svg(decoded)

    # If no closing tag found, try to repair truncated SVG
    if not svg and "<svg" in decoded:
        # Close any open tags and force close the SVG
        truncated = decoded.strip()
        if not truncated.endswith("</svg>"):
            # Close open path/element and svg
            truncated = truncated.rstrip('"/ ')
            truncated += '"/></svg>'
        svg = extract_svg(truncated)

    if svg:
        svg = fix_svg_canvas(svg)

    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)

    return svg

In [ ]:
# ── Quick sanity check ──
for p in ["a red circle", "a green tree with brown trunk", "a simple blue bird icon"]:
    svg = generate_svg(p)
    print(f"{p} → Valid: {is_valid_svg(svg)}, Length: {len(svg)} chars")
    print(f"  {svg}...\n")

a red circle → Valid: True, Length: 608 chars
  <svg fill="none" height="256" viewBox="0 0 256 256" width="256" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink"><clipPath id="a"><path d="m0 0h128v128h-128z"/></clipPath><g clip-path="url(#a)"><path d="m128 64c0 35.34-28.66 64-64 64s-64-28.66-64-64 28.66-64 64-64 64 28.66 64 64" stroke="#000000" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"/><path d="m64 10.67c-33.3 0-60.67 27.37-60.67 60.67s27.37 60.67 60.67 60.67 60.67-27.37 60.67-60.67-27.37-60.67-60.67-60.67" stroke="#ff0000" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"/></g></svg>...

a green tree with brown trunk → Valid: True, Length: 210 chars
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="80" fill="hsl(349, 60%, 50%)"/></svg>...

a simple blue bird icon → Valid: True, Length: 393 chars
  

In [ ]:
# Debug: generate without fallback to see what's failing
test_prompts_debug = test_df["prompt"].head(10).tolist()

for p in test_prompts_debug:
    chat_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{p}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    input_ids = text_tokenizer.encode(chat_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    eos_ids = text_tokenizer.encode("<|im_end|>", add_special_tokens=False)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            eos_token_id=eos_ids,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    decoded = text_tokenizer.decode(new_tokens, skip_special_tokens=True)
    svg = extract_svg(decoded)
    valid = is_valid_svg(svg)

    print(f"Prompt: {p[:60]}...")
    print(f"  Tokens: {len(new_tokens)}, SVG found: {bool(svg)}, Valid: {valid}, Len: {len(svg) if svg else 0}")
    if not valid and decoded:
        print(f"  Raw start: {decoded[:150]}")
    print()

Prompt: firewood stack cut logs wood with leaf illustration....
  Tokens: 512, SVG found: False, Valid: False, Len: 0
  Raw start: <svg fill="none" height="128" viewBox="0 0 128 128" width="128" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink"><clipPat

Prompt: The image shows five horizontal lines of varying thicknesses...
  Tokens: 512, SVG found: False, Valid: False, Len: 0
  Raw start: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#333333" fill-opacity="1.0"  fillin

Prompt: A stylized icon depicting a curved arrow within a square sha...
  Tokens: 512, SVG found: False, Valid: False, Len: 0
  Raw start: <svg height="128" viewBox="0 0 256 256" width="128" xmlns="http://www.w3.org/2000/svg"><g fill="#333333"><path d="m216 128-128-128"/><path d="m248 128

Prompt: The image contains black geometric shapes against a white ba...
  Tokens: 512, SVG found: False, Valid: False, Len: 0
  Raw sta

## 8. Generate Submission (1000 SVGs)

In [ ]:
test_df = pd.read_csv(CONFIG["test_prompts_csv"])
print(f"Test prompts: {len(test_df)}")
test_df.head()

Test prompts: 1000


,id,prompt
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...
2,ea045c7a247166f061ce504d9b7ccaab,A stylized icon depicting a curved arrow withi...
3,8fe82f3af89e487b31236ca829c3f071,The image contains black geometric shapes agai...
4,600464e4d92c75338462271a09b3f176,The image shows a single dark gray triangle po...


In [ ]:
rows = []
invalid_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = row["prompt"]
    svg = generate_svg(prompt)

    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(prompt)

    if len(svg) > CONFIG["max_svg_chars"]:
        svg = fallback_svg(prompt)
        invalid_count += 1

    rows.append({"id": row["id"], "svg": svg})

    if (idx + 1) % 1 == 0:
        elapsed = time.time() - t0
        rate = (idx + 1) / elapsed
        remaining = (len(test_df) - idx - 1) / rate
        print(f"  [{idx+1}/{len(test_df)}] {rate:.1f} samples/s, "
              f"~{remaining/60:.1f} min remaining, {invalid_count} fallbacks")

elapsed_min = (time.time() - t0) / 60
print(f"\nDone! {len(rows)} SVGs in {elapsed_min:.1f} min")
print(f"Fallbacks: {invalid_count} ({100*invalid_count/len(rows):.1f}%)")

  [1/1000] 0.0 samples/s, ~4022.4 min remaining, 0 fallbacks
  [2/1000] 0.0 samples/s, ~3015.3 min remaining, 0 fallbacks
  [3/1000] 0.0 samples/s, ~2135.0 min remaining, 0 fallbacks
  [4/1000] 0.0 samples/s, ~1684.4 min remaining, 0 fallbacks
  [5/1000] 0.0 samples/s, ~1539.9 min remaining, 0 fallbacks
  [6/1000] 0.0 samples/s, ~1449.6 min remaining, 0 fallbacks
  [7/1000] 0.0 samples/s, ~1674.5 min remaining, 0 fallbacks
  [8/1000] 0.0 samples/s, ~1748.6 min remaining, 0 fallbacks
  [9/1000] 0.0 samples/s, ~1835.9 min remaining, 0 fallbacks
  [10/1000] 0.0 samples/s, ~1781.7 min remaining, 0 fallbacks
  [11/1000] 0.0 samples/s, ~1661.9 min remaining, 0 fallbacks


KeyboardInterrupt: 

In [ ]:
def generate_svgs_batched(prompts, max_new_tokens=2048, batch_size=8):
    eos_ids = text_tokenizer.encode("<|im_end|>", add_special_tokens=False)
    text_tokenizer.padding_side = "left"
    if text_tokenizer.pad_token is None:
        text_tokenizer.pad_token = text_tokenizer.eos_token

    results = []
    t_start = time.time()

    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]

        chat_texts = [
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{p}<|im_end|>\n"
            f"<|im_start|>assistant\n"
            for p in batch_prompts
        ]

        inputs = text_tokenizer(chat_texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=CONFIG["temperature"],
                top_p=CONFIG["top_p"],
                repetition_penalty=CONFIG["repetition_penalty"],
                eos_token_id=eos_ids,
            )

        for j, ids in enumerate(output_ids):
            input_len = inputs["input_ids"].shape[1]
            new_tokens = ids[input_len:]
            decoded = text_tokenizer.decode(new_tokens, skip_special_tokens=True)

            svg = extract_svg(decoded)
            if not svg and "<svg" in decoded:
                truncated = decoded.strip()
                if not truncated.endswith("</svg>"):
                    truncated = truncated.rstrip('"/ ') + '"/></svg>'
                svg = extract_svg(truncated)
            if svg:
                svg = fix_svg_canvas(svg)
            if not is_valid_svg(svg):
                svg = fallback_svg(batch_prompts[j])
            results.append(svg)

        elapsed = time.time() - t_start
        done = len(results)
        rate = done / elapsed
        remaining = (len(prompts) - done) / rate / 60
        print(f"  [{done}/{len(prompts)}] {rate:.2f} samples/s, ~{remaining:.1f} min left", flush=True)

    return results


# Run submission
t0 = time.time()
all_prompts = test_df["prompt"].tolist()
all_svgs = generate_svgs_batched(all_prompts, max_new_tokens=2048, batch_size=8)

rows = []
invalid_count = 0
for idx, (_, row) in enumerate(test_df.iterrows()):
    svg = all_svgs[idx]
    if len(svg) > CONFIG["max_svg_chars"]:
        svg = fallback_svg(row["prompt"])
        invalid_count += 1
    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(row["prompt"])
    rows.append({"id": row["id"], "svg": svg})

print(f"\nDone! {len(rows)} SVGs in {(time.time()-t0)/60:.1f} min")
print(f"Fallbacks: {invalid_count}")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(CONFIG["submission_path"], index=False)
print(f"Saved: {CONFIG['submission_path']}")

  [8/1000] 0.03 samples/s, ~600.8 min left
  [16/1000] 0.03 samples/s, ~595.7 min left
  [24/1000] 0.03 samples/s, ~590.8 min left
  [32/1000] 0.03 samples/s, ~585.3 min left
  [40/1000] 0.03 samples/s, ~580.1 min left
  [48/1000] 0.03 samples/s, ~574.9 min left
  [56/1000] 0.03 samples/s, ~570.1 min left
  [64/1000] 0.03 samples/s, ~565.9 min left
  [72/1000] 0.03 samples/s, ~560.7 min left
  [80/1000] 0.03 samples/s, ~555.6 min left
  [88/1000] 0.03 samples/s, ~550.6 min left
  [96/1000] 0.03 samples/s, ~545.6 min left
  [104/1000] 0.03 samples/s, ~540.6 min left
  [112/1000] 0.03 samples/s, ~535.6 min left
  [120/1000] 0.03 samples/s, ~530.5 min left
  [128/1000] 0.03 samples/s, ~525.7 min left
  [136/1000] 0.03 samples/s, ~521.0 min left
  [144/1000] 0.03 samples/s, ~516.2 min left
  [152/1000] 0.03 samples/s, ~511.3 min left
  [160/1000] 0.03 samples/s, ~506.6 min left
  [168/1000] 0.03 samples/s, ~502.0 min left
  [176/1000] 0.03 samples/s, ~497.2 min left
  [184/1000] 0.03 sampl

In [ ]:
sub_df = pd.DataFrame(rows)
sub_df["svg_len"] = sub_df["svg"].str.len()

print(f"SVG length stats:")
print(sub_df["svg_len"].describe())
assert sub_df["svg_len"].max() <= CONFIG["max_svg_chars"], "SVGs exceed char limit!"
#assert len(sub_df) == len(test_df), "Row count mismatch!"

sub_df[["id", "svg"]].to_csv(CONFIG["submission_path"], index=False)
print(f"\nSaved: {CONFIG['submission_path']} ({len(sub_df)} rows)")

SVG length stats:
count    1000.000000
mean     1069.251000
std       807.383764
min       123.000000
25%       347.750000
50%       739.500000
75%      2124.500000
max      4253.000000
Name: svg_len, dtype: float64

Saved: /content/submission.csv (1000 rows)


In [ ]:
# ── Download submission from Colab ──
from google.colab import files
files.download(CONFIG["submission_path"])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Save Model (Optional)

In [ ]:
# ── Save LoRA adapter to Google Drive ──
SAVE_TO_DRIVE = True  # Set True when ready

if SAVE_TO_DRIVE:
    import shutil
    drive_path = "/content/drive/MyDrive/qwen2b_svg_lora"
    shutil.copytree(CONFIG["output_dir"], drive_path, dirs_exist_ok=True)
    print(f"LoRA adapter copied to Google Drive: {drive_path}")

LoRA adapter copied to Google Drive: /content/drive/MyDrive/qwen2b_svg_lora


In [ ]:
# ── Push to HuggingFace (optional) ──
PUSH_TO_HF = False

if PUSH_TO_HF:
    HF_USERNAME = "YOUR_USERNAME"
    HF_TOKEN = "YOUR_HF_TOKEN"
    model.push_to_hub(f"{HF_USERNAME}/qwen2b-svg-lora", token=HF_TOKEN)
    tokenizer.push_to_hub(f"{HF_USERNAME}/qwen2b-svg-lora", token=HF_TOKEN)
    print("Pushed to HuggingFace Hub.")

In [ ]:
# ── Save merged model for Kaggle offline inference ──
SAVE_MERGED = True

if SAVE_MERGED:
    merged_dir = "/content/qwen2b_svg_merged"
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to: {merged_dir}")
    print("Upload this as a Kaggle dataset for offline inference.")
    import shutil
    drive_path = "/content/drive/MyDrive/merged"
    shutil.copytree(CONFIG["output_dir"], drive_path, dirs_exist_ok=True)
    print(f"Merged copied to Google Drive: {drive_path}")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `/content/qwen2b_svg_merged`:   0%|          | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `/content/qwen2b_svg_merged`: 100%|██████████| 1/1 [00:04<00:00,  4.56s/it]


Successfully copied all 1 files from cache to `/content/qwen2b_svg_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7397.36it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:17<00:00, 17.68s/it]


Unsloth: Merge process complete. Saved to `/content/qwen2b_svg_merged`
Merged model saved to: /content/qwen2b_svg_merged
Upload this as a Kaggle dataset for offline inference.
Merged copied to Google Drive: /content/drive/MyDrive/merged
